# Search Service Playground (`search_async`)

Interactive playground for experimenting with `SearchService.search_async(...)` against Neo4j vector embeddings.

**Quick Start:**
1. Run **Section 1** once to initialize Neo4j and the search services.
2. Edit `QUERY` (and any optional filter/hydration/display variables) in **Section 2** — no need to modify the `search_async(...)` call arguments.
3. Run **Sections 3 & 4** to resolve the target `Group` (defaults to `"GoSmart Health"`), execute `search_async`, and view tabularized results.
4. Run **Section 5** to demonstrate graph traversal from matched nodes back to `:Encounter`, `:FHIR_Patient`, and `:ImagingStudy` nodes.
5. Optionally run **Section 6** to compare multiple query phrases in a single table, or **Section 7** to inspect the full JSON payload of a specific hit.

## 1. Environment & Service Setup

In [9]:
import sys
import time
from pathlib import Path
from typing import Optional, Sequence, Union

from rich.console import Console
from rich.table import Table

# Resolve project root whether Jupyter is launched from the repo root or notebooks/
_cwd = Path.cwd().resolve()
PROJECT_ROOT = _cwd if (_cwd / "src").is_dir() else _cwd.parent
SRC_DIR = PROJECT_ROOT / "src"
for _p in (SRC_DIR, PROJECT_ROOT):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from gosmart_graphite.core.config import Settings
from gosmart_graphite.db.neo4j import Neo4jClient
from gosmart_graphite.model.embedding import EMBEDDING_VECTOR_INDEX_NAME
from gosmart_graphite.model.search import SearchIncludeLabel, SearchResult
from gosmart_graphite.service.embedding_service import DEFAULT_EMBEDDING_TAG, EmbeddingService
from gosmart_graphite.service.group_service import DEFAULT_GROUP_NAME, DEFAULT_MY_GROUP_MAX_DEPTH, GroupService
from gosmart_graphite.service.llm_config_service import LLMConfigService
from gosmart_graphite.service.search_service import SearchService

env_file = PROJECT_ROOT / ".env"
settings = Settings()

neo4j_client = Neo4jClient(settings=settings)
group_svc = GroupService(client=neo4j_client)
llm_cfg_svc = LLMConfigService(client=neo4j_client)
embedding_svc = EmbeddingService(llm_config_service=llm_cfg_svc, settings=settings)
search_svc = SearchService(embedding_service=embedding_svc, client=neo4j_client)

connected = await neo4j_client.verify_connectivity()
print(f"Project Root : {PROJECT_ROOT}")
print(f"Neo4j URI    : {neo4j_client.settings.DB_CONNECTION_URI}")
print(f"Neo4j DB     : {neo4j_client.settings.DB_DATABASE}")
print(f"Vector Index : {EMBEDDING_VECTOR_INDEX_NAME}")
print(f"Connected    : {connected}")
if not connected:
    raise RuntimeError("Unable to connect to Neo4j. Verify GOSMART_GS_DB_* / NEO4J_* settings in .env.")

2026-11-23 10:31:57 [info     ] nlu_models_loaded              intents=['find_study', 'get_report'] nlu_model_path=/Users/manabutokunaga/development/dicom-fhir/gosmart-graphite/models/intent_models.zip spacy_version=3.8.18
Project Root : /Users/manabutokunaga/development/dicom-fhir/gosmart-graphite
Neo4j URI    : neo4j+s://e2d8e1dd.databases.neo4j.io
Neo4j DB     : e2d8e1dd
Vector Index : embedding_vector_idx
Connected    : True


## 2. Search Parameters Configuration

Modify any variable in this block to experiment with `search_async` without touching the method call arguments below.

In [10]:
# --- Query Phrase (edit or uncomment a sample phrase to experiment) ---
QUERY: str = "acute onset shortness of breath"
# QUERY = "bilateral pulmonary embolism"
# QUERY = "clear lungs with normal cardiac silhouette"
# QUERY = "right lower lobe pneumonia"

# --- Group Scope ---
# Defaults to the Group node with name = "GoSmart Health".
# Set to None to run an unscoped search across all groups.
GROUP_NAME: Optional[str] = DEFAULT_GROUP_NAME  # "GoSmart Health"

# --- Core SearchService.search_async Parameters ---
LIMIT: int = 10                                    # Service default: DEFAULT_SEARCH_LIMIT (20)
MIN_SCORE: Optional[float] = None                  # Minimum cosine similarity in [0.0, 1.0], e.g. 0.75
TAG: str = DEFAULT_EMBEDDING_TAG                   # Embedding LLM_Config tag (default: "embedding-v1")

# Opt-in related clinical/DICOM context hydration (None = base SearchResult fields only)
INCLUDE: Optional[Sequence[Union[SearchIncludeLabel, str]]] = None
# INCLUDE = [
#     SearchIncludeLabel.ENCOUNTER,
#     SearchIncludeLabel.FHIR_PATIENT,
#     SearchIncludeLabel.IMAGING_STUDY,
#     SearchIncludeLabel.PATIENT,
#     SearchIncludeLabel.STUDY,
# ]

# Optional target node label and embedding field filters
TARGET_LABELS: Optional[Sequence[str]] = None
# TARGET_LABELS = ["DiagnosticReport"]

TARGET_FIELD_TAGS: Optional[Sequence[str]] = None
# TARGET_FIELD_TAGS = [DiagnosticReportFieldTag.FINDINGS, DiagnosticReportFieldTag.CONCLUSION]

INCLUDE_DELETED: bool = False                      # Set True to include soft-deleted nodes/embeddings
TEST_ID: Optional[str] = None                      # None = production data; set string for isolated test data
MAX_GROUP_DEPTH: int = DEFAULT_MY_GROUP_MAX_DEPTH  # Max upward hops when resolving owning Group (default: 8)

# --- Table Display Options ---
CONSOLE_WIDTH: int = 160                           # Rich table console width for wider Text previews
SHORT_NODE_ID: bool = True                         # Show 8-char UUID prefix in tables (full ID in Section 7)
TEXT_PREVIEW_CHARS: Optional[int] = 220            # Truncate matched text in table; set None for full text

## 3. Resolve Target Group (`GoSmart Health`) & Active Embedding Config

When `GROUP_NAME` is configured, retrieves the active `:Group` node (default `name = "GoSmart Health"`) and extracts its deterministic `id` for `group_id`. Also inspects the active embedding `LLM_Config` resolved for `(TAG, group_id)`.

In [3]:
console = Console(width=CONSOLE_WIDTH)

group_node = None
group_id: Optional[str] = None

if GROUP_NAME and GROUP_NAME.strip():
    group_node = await group_svc.resolve_target_group(identifier=GROUP_NAME, test_id=TEST_ID)
    group_id = group_node["id"]

llm_cfg = await llm_cfg_svc.resolve_config_for_group(tag=TAG, group_id=group_id, test_id=TEST_ID)

ctx_table = Table(title="Active Search Scope & Embedding Configuration", show_lines=False)
ctx_table.add_column("Setting", style="cyan", no_wrap=True)
ctx_table.add_column("Value", style="white")

if group_node:
    ctx_table.add_row("Group Name", str(group_node.get("name", "-")))
    ctx_table.add_row("Group Code", str(group_node.get("code", "-")))
    ctx_table.add_row("Group Type", str(group_node.get("group_type", "-")))
    ctx_table.add_row("Group ID", str(group_id))
else:
    ctx_table.add_row("Group Scope", "Unscoped (group_id=None)")

if llm_cfg:
    ctx_table.add_row("Embedding Tag", str(llm_cfg.tag))
    ctx_table.add_row("Provider", str(llm_cfg.provider))
    ctx_table.add_row("Model", str(llm_cfg.model))
    ctx_table.add_row("Dimensions", str(llm_cfg.dimensions or "-"))
    ctx_table.add_row("Endpoint URL", str(llm_cfg.url or "-"))
else:
    ctx_table.add_row("Embedding Config", f"[bold red]Not found for tag='{TAG}'[/bold red]")

console.print(ctx_table)

     Active Search Scope & Embedding Configuration      
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Setting       ┃ Value                                ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Group Name    │ GoSmart Health                       │
│ Group Code    │ GOSMART_HEALTH                       │
│ Group Type    │ HEALTH_SYSTEM                        │
│ Group ID      │ 09951e9a-8891-5da2-9a44-ff535d5ab710 │
│ Embedding Tag │ embedding-v1                         │
│ Provider      │ ollama                               │
│ Model         │ embeddinggemma-2:270m                │
│ Dimensions    │ 768                                  │
│ Endpoint URL  │ http://m5.local.:11434               │
└───────────────┴──────────────────────────────────────┘

## 4. Run `search_async` & Tabularize Results

Executes `search_svc.search_async(...)` using exclusively the variables defined in Section 2 (plus resolved `group_id`) and renders the results as a Rich table.

In [4]:
def _truncate(text: str, max_chars: Optional[int]) -> str:
    clean = " ".join((text or "").split())
    if max_chars is not None and len(clean) > max_chars:
        return clean[: max_chars - 3].rstrip() + "..."
    return clean


def _format_node_id(node_id: str, short: bool) -> str:
    if short and len(node_id) > 8:
        return f"{node_id[:8]}…"
    return node_id


def _normalize_include_set(include: Optional[Sequence[Union[SearchIncludeLabel, str]]]) -> set[str]:
    if not include:
        return set()
    return {item.value if isinstance(item, SearchIncludeLabel) else str(item).strip() for item in include}


def _summarize_encounter(search_hit: SearchResult) -> str:
    if not search_hit.encounter:
        return "-"
    parts = [search_hit.encounter.resource_id]
    if search_hit.encounter.status:
        parts.append(f"({search_hit.encounter.status})")
    return " ".join(parts)


def _summarize_fhir_patient(search_hit: SearchResult) -> str:
    if not search_hit.fhir_patient:
        return "-"
    name = search_hit.fhir_patient.name[0] if search_hit.fhir_patient.name else None
    return f"{name} [{search_hit.fhir_patient.resource_id}]" if name else search_hit.fhir_patient.resource_id


def _summarize_imaging_studies(search_hit: SearchResult) -> str:
    if not search_hit.imaging_studies:
        return "-"
    ids = [img.resource_id for img in search_hit.imaging_studies]
    return f"{len(ids)}: {', '.join(ids[:3])}{'...' if len(ids) > 3 else ''}"


def _summarize_dicom_patient(search_hit: SearchResult) -> str:
    if not search_hit.patient:
        return "-"
    if search_hit.patient.patient_name:
        return f"{search_hit.patient.patient_name} ({search_hit.patient.patient_id})"
    return search_hit.patient.patient_id


def _summarize_dicom_studies(search_hit: SearchResult) -> str:
    if not search_hit.studies:
        return "-"
    labels = [st.accession_number or st.study_instance_uid for st in search_hit.studies]
    return f"{len(labels)}: {', '.join(labels[:2])}{'...' if len(labels) > 2 else ''}"


def display_search_results(
    query: str,
    search_hits: list[SearchResult],
    *,
    elapsed_millis: float,
    include: Optional[Sequence[Union[SearchIncludeLabel, str]]] = None,
    include_deleted: bool = False,
    text_preview_chars: Optional[int] = 220,
    short_node_id: bool = True,
    console_width: int = 160,
) -> None:
    out = Console(width=console_width)
    include_set = _normalize_include_set(include)
    if not search_hits:
        out.print(
            f"[yellow]No results found for query '{query}' ({elapsed_millis:.1f} ms). "
            "Check GROUP_NAME, TAG, MIN_SCORE, or ensure embeddings exist in Neo4j.[/yellow]"
        )
        return

    table = Table(
        title=f"Search Results for '{query}' ({len(search_hits)} hit{'s' if len(search_hits) != 1 else ''}, {elapsed_millis:.1f} ms)",
        show_lines=True,
        expand=True,
    )
    table.add_column("#", justify="right", style="cyan", no_wrap=True)
    table.add_column("Score", justify="right", style="green", no_wrap=True)
    table.add_column("Label", style="magenta", no_wrap=True)
    table.add_column("Field Tag", style="yellow", no_wrap=True)
    table.add_column("Node ID", style="dim", no_wrap=True)
    table.add_column("Text", style="white", ratio=3)

    if include_deleted:
        table.add_column("Deleted", justify="center", style="red", no_wrap=True)
    if SearchIncludeLabel.ENCOUNTER.value in include_set:
        table.add_column("Encounter", style="bright_blue")
    if SearchIncludeLabel.FHIR_PATIENT.value in include_set:
        table.add_column("FHIR Patient", style="bright_cyan")
    if SearchIncludeLabel.IMAGING_STUDY.value in include_set:
        table.add_column("Imaging Studies", style="bright_magenta")
    if SearchIncludeLabel.PATIENT.value in include_set:
        table.add_column("DICOM Patient", style="bright_green")
    if SearchIncludeLabel.STUDY.value in include_set:
        table.add_column("DICOM Studies", style="bright_yellow")

    for the_rank, the_hit in enumerate(search_hits, start=1):
        row = [
            str(the_rank),
            f"{the_hit.score:.4f}",
            the_hit.node_label or "-",
            the_hit.target_field_tag or "-",
            _format_node_id(the_hit.node_id, short_node_id),
            _truncate(the_hit.text, text_preview_chars),
        ]
        if include_deleted:
            row.append("YES" if the_hit.is_deleted else "no")
        if SearchIncludeLabel.ENCOUNTER.value in include_set:
            row.append(_summarize_encounter(the_hit))
        if SearchIncludeLabel.FHIR_PATIENT.value in include_set:
            row.append(_summarize_fhir_patient(the_hit))
        if SearchIncludeLabel.IMAGING_STUDY.value in include_set:
            row.append(_summarize_imaging_studies(the_hit))
        if SearchIncludeLabel.PATIENT.value in include_set:
            row.append(_summarize_dicom_patient(the_hit))
        if SearchIncludeLabel.STUDY.value in include_set:
            row.append(_summarize_dicom_studies(the_hit))
        table.add_row(*row)

    out.print(table)


t0 = time.perf_counter()
results = await search_svc.search_async(
    query=QUERY,
    group_id=group_id,
    limit=LIMIT,
    min_score=MIN_SCORE,
    tag=TAG,
    include=INCLUDE,
    target_labels=TARGET_LABELS,
    target_field_tags=TARGET_FIELD_TAGS,
    include_deleted=INCLUDE_DELETED,
    test_id=TEST_ID,
    max_group_depth=MAX_GROUP_DEPTH,
)
elapsed_ms = (time.perf_counter() - t0) * 1000.0

display_search_results(
    QUERY,
    results,
    elapsed_millis=elapsed_ms,
    include=INCLUDE,
    include_deleted=INCLUDE_DELETED,
    text_preview_chars=TEXT_PREVIEW_CHARS,
    short_node_id=SHORT_NODE_ID,
    console_width=CONSOLE_WIDTH,
)

2026-11-23 10:31:49 [info     ] Embedding text: task: search result | query: acute onset shortness of breath
2026-11-23 10:31:49 [debug    ] initializing_genkit_ollama     base_url=http://m5.local.:11434 tag=embedding-v1


                                           Search Results for 'acute onset shortness of breath' (10 hits, 2720.9 ms)                                            
┏━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃  # ┃  Score ┃ Label            ┃ Field Tag  ┃ Node ID   ┃ Text                                                                                               ┃
┡━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│  1 │ 0.8500 │ DiagnosticReport │ CONCLUSION │ f9bc8f7b… │ 2001-12-01 # Chief Complaint - Fatigue - Thirst - Chest Pressure - Nasal Congestion # History of   │
│    │        │                  │            │           │ Present Illness Dewey930 is a 75 year-old nonhispanic white male. # Social History Patient is      │
│    │        │                  │            │           │ single. Patient quit smoki...                                                                      │
├────┼────────┼──────────────────┼────────────┼───────────┼────────────────────────────────────────────────────────────────────────────────────────────────────┤
│  2 │ 0.8489 │ DiagnosticReport │ CONCLUSION │ 402262a3… │ 2014-12-03 # Chief Complaint - Shortness of Breath - Cough - Reduced Sex Drive - urinary frequency │
│    │        │                  │            │           │ # History of Present Illness Waylon572 is a 55 year-old nonhispanic white male. # Social History   │
│    │        │                  │            │           │ Patient is single. Pa...                                                                           │
├────┼────────┼──────────────────┼────────────┼───────────┼────────────────────────────────────────────────────────────────────────────────────────────────────┤
│  3 │ 0.8470 │ DiagnosticReport │ CONCLUSION │ 54f5d36f… │ 2016-07-08 # Chief Complaint No complaints. # History of Present Illness Florrie453 Aja848 is a 81 │
│    │        │                  │            │           │ year-old nonhispanic white female. Patient has a history of gingival disease (disorder),           │
│    │        │                  │            │           │ medication review due (situat...                                                                   │
├────┼────────┼──────────────────┼────────────┼───────────┼────────────────────────────────────────────────────────────────────────────────────────────────────┤
│  4 │ 0.8452 │ DiagnosticReport │ CONCLUSION │ d94ca155… │ 1993-12-17 # Chief Complaint No complaints. # History of Present Illness Rosario163 Saundra460 is  │
│    │        │                  │            │           │ a 27 year-old nonhispanic native female. Patient has a history of medication review due            │
│    │        │                  │            │           │ (situation), acute viral pharyn...                                                                 │
├────┼────────┼──────────────────┼────────────┼───────────┼────────────────────────────────────────────────────────────────────────────────────────────────────┤
│  5 │ 0.8450 │ DiagnosticReport │ CONCLUSION │ b5821709… │ 2015-11-19 # Chief Complaint No complaints. # History of Present Illness Florrie453 Aja848 is a 80 │
│    │        │                  │            │           │ year-old nonhispanic white female. Patient has a history of gingival disease (disorder),           │
│    │        │                  │            │           │ medication review due (situat...                                                                   │
├────┼────────┼──────────────────┼────────────┼───────────┼────────────────────────────────────────────────────────────────────────────────────────────────────┤
│  6 │ 0.8442 │ DiagnosticReport │ CONCLUSION │ fca4bb8e… │ 2000-02-18 # Chief Complaint - Fatigue - Thirst - Tingling in Hands and Feet

## 5. Context Traversal Demo (`Encounter`, `FHIR_Patient`, `ImagingStudy`)

Demonstrates opt-in post-`LIMIT` graph traversal via `include=[SearchIncludeLabel.ENCOUNTER, SearchIncludeLabel.FHIR_PATIENT, SearchIncludeLabel.IMAGING_STUDY]`:
- **`:Encounter`**: `(target)-[:PERFORMED_IN|HAS_ENCOUNTER]->(:Encounter)`
- **`:FHIR_Patient`**: `(target)-[:SUBJECT]->(:FHIR_Patient)` or `(target)-[:PERFORMED_IN|HAS_ENCOUNTER]->(:Encounter)-[:SUBJECT]->(:FHIR_Patient)`
- **`:ImagingStudy`**: `(target)-[:BASED_ON_STUDY]->(:ImagingStudy)` or `(target)-[:PERFORMED_IN|HAS_ENCOUNTER]->(:Encounter)<-[:PERFORMED_IN]-(:ImagingStudy)`

In [5]:
from datetime import date
from typing import Any

# --- Traversal Demo Variables ---
TRAVERSAL_QUERY: str = QUERY  # Reuses Section 2 QUERY by default (or override with a custom phrase)
TRAVERSAL_LIMIT: int = 5
TRAVERSAL_INCLUDE: list[SearchIncludeLabel] = [
    SearchIncludeLabel.ENCOUNTER,
    SearchIncludeLabel.FHIR_PATIENT,
    SearchIncludeLabel.IMAGING_STUDY,
]


def _short_macro(value: Optional[str]) -> str:
    if not value:
        return "-"
    return value.split("|")[-1].strip() or value


def _fmt_date(dt: Optional[Union[date, str]]) -> str:
    if dt is None:
        return "-"
    if isinstance(dt, date):
        return dt.strftime("%Y-%m-%d")
    return dt[:10]


def _format_encounter_detail(search_hit: SearchResult) -> str:
    enc = search_hit.encounter
    if enc is None:
        return "[dim]-[/dim]"
    lines = [f"id: {enc.resource_id}"]
    meta: list[Any] = [p for p in (enc.status, _short_macro(enc.class_) if enc.class_ else None) if p]
    if meta:
        lines.append(f"status/class: {' / '.join(meta)}")
    if enc.periodStart:
        lines.append(f"start: {_fmt_date(enc.periodStart)}")
    if enc.type:
        lines.append(f"type: {_truncate(_short_macro(enc.type[0]), 48)}")
    return "\n".join(lines)


def _format_fhir_patient_detail(search_hit: SearchResult) -> str:
    pat = search_hit.fhir_patient
    if pat is None:
        return "[dim]-[/dim]"
    lines = [f"id: {pat.resource_id}"]
    if pat.name:
        lines.append(f"name: {pat.name[0]}")
    demo: list[Any] = [p for p in (pat.gender, pat.birthDate) if p]
    if demo:
        lines.append(f"demo: {' / '.join(demo)}")
    return "\n".join(lines)


def _format_imaging_studies_detail(search_hit: SearchResult) -> str:
    studies = search_hit.imaging_studies or []
    if not studies:
        return "[dim]-[/dim]"
    blocks: list[str] = []
    for img in studies:
        modalities = list(dict.fromkeys(_short_macro(m) for m in (img.seriesModality or []) if m))
        header = f"id: {img.resource_id}" + (f" ({img.status})" if img.status else "")
        details = [header]
        if img.started:
            details.append(f"started: {_fmt_date(img.started)}")
        if modalities:
            details.append(f"modality: {', '.join(modalities)}")
        if img.procedureCodeText:
            details.append(f"proc: {_truncate(img.procedureCodeText, 48)}")
        if img.studyUid:
            details.append(f"studyUid: {_truncate(img.studyUid, 36)}")
        blocks.append("\n".join(details))
    return "\n---\n".join(blocks)


t_trav = time.perf_counter()
hydrated_results = await search_svc.search_async(
    query=TRAVERSAL_QUERY,
    group_id=group_id,
    limit=TRAVERSAL_LIMIT,
    min_score=MIN_SCORE,
    tag=TAG,
    include=TRAVERSAL_INCLUDE,
    target_labels=TARGET_LABELS,
    target_field_tags=TARGET_FIELD_TAGS,
    include_deleted=INCLUDE_DELETED,
    test_id=TEST_ID,
    max_group_depth=MAX_GROUP_DEPTH,
)
trav_ms = (time.perf_counter() - t_trav) * 1000.0

# 1) Standard search table with hydrated context summary columns
display_search_results(
    TRAVERSAL_QUERY,
    hydrated_results,
    elapsed_millis=trav_ms,
    include=TRAVERSAL_INCLUDE,
    include_deleted=INCLUDE_DELETED,
    text_preview_chars=TEXT_PREVIEW_CHARS,
    short_node_id=SHORT_NODE_ID,
    console_width=CONSOLE_WIDTH,
)

# 2) Detailed traversal breakdown: Matched Target -> Encounter / FHIR_Patient / ImagingStudy
trav_console = Console(width=CONSOLE_WIDTH)
if hydrated_results:
    detail_table = Table(
        title=f"FHIR Context Traversal Breakdown for '{TRAVERSAL_QUERY}' ({len(hydrated_results)} hits)",
        show_lines=True,
        expand=True,
    )
    detail_table.add_column("#", justify="right", style="cyan", no_wrap=True)
    detail_table.add_column("Score", justify="right", style="green", no_wrap=True)
    detail_table.add_column("Matched Target", style="magenta", ratio=1)
    detail_table.add_column("Encounter", style="bright_blue", ratio=1)
    detail_table.add_column("FHIR_Patient", style="bright_cyan", ratio=1)
    detail_table.add_column("ImagingStudy", style="bright_magenta", ratio=2)

    for rank, hit in enumerate(hydrated_results, start=1):
        target_summary = (
            f"{hit.node_label or '-'}\n"
            f"id: {_format_node_id(hit.node_id, SHORT_NODE_ID)}\n"
            f"tag: {hit.target_field_tag or '-'}"
        )
        detail_table.add_row(
            str(rank),
            f"{hit.score:.4f}",
            target_summary,
            _format_encounter_detail(hit),
            _format_fhir_patient_detail(hit),
            _format_imaging_studies_detail(hit),
        )

    trav_console.print(detail_table)

    n_enc = sum(1 for h in hydrated_results if h.encounter is not None)
    n_pat = sum(1 for h in hydrated_results if h.fhir_patient is not None)
    n_img_hits = sum(1 for h in hydrated_results if h.imaging_studies)
    n_img_total = sum(len(h.imaging_studies or []) for h in hydrated_results)
    trav_console.print(
        f"[bold]Traversal Coverage ({len(hydrated_results)} hits):[/bold] "
        f"Encounter={n_enc}, FHIR_Patient={n_pat}, ImagingStudy={n_img_hits} hits ({n_img_total} total studies)"
    )

2026-11-23 10:31:52 [info     ] Embedding text: task: search result | query: acute onset shortness of breath


                                            Search Results for 'acute onset shortness of breath' (5 hits, 1883.0 ms)                                            
┏━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ # ┃  Score ┃ Label            ┃ Field Tag  ┃ Node ID   ┃ Text         ┃ Encounter         ┃ FHIR Patient                                 ┃ Imaging Studies   ┃
┡━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ 1 │ 0.8500 │ DiagnosticReport │ CONCLUSION │ f9bc8f7b… │ 2001-12-01 # │ 128950 (finished) │ Mr. Dewey930 Carlos172 Nader710 PhD [128861] │ 2: 128952, 128953 │
│   │        │                  │            │           │ Chief        │                   │                                              │                   │
│   │        │                  │            │           │ Complaint -  │                   │                                              │                   │
│   │        │                  │            │           │ Fatigue -    │                   │                                              │                   │
│   │        │                  │            │           │ Thirst -     │                   │                                              │                   │
│   │        │                  │            │           │ Chest        │                   │                                              │                   │
│   │        │                  │            │           │ Pressure -   │                   │                                              │                   │
│   │        │                  │            │           │ Nasal        │                   │                                              │                   │
│   │        │                  │            │           │ Congestion # │                   │                                              │                   │
│   │        │                  │            │           │ History of   │                   │                                              │                   │
│   │        │                  │            │           │ Present      │                   │                                              │                   │
│   │        │                  │            │           │ Illness      │                   │                                              │                   │
│   │        │                  │            │           │ Dewey930 is  │                   │                                              │                   │
│   │        │                  │            │           │ a 75         │                   │                                              │                   │
│   │        │                  │            │           │ year-old     │                   │                                              │                   │
│   │        │                  │            │           │ nonhispanic  │                   │                                              │                   │
│   │        │                  │            │           │ white male.  │                   │                                              │                   │
│   │        │                  │            │           │ # Social     │                   │                                              │                   │
│   │        │                  │            │           │ History      │                   │                                              │                   │
│   │        │                  │            │           │ Patient is   │                   │                                              │                   │
│   │        │                  │            │           │ single.      │                   │                                           

                                        FHIR Context Traversal Breakdown for 'acute onset shortness of breath' (5 hits)                                         
┏━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ # ┃  Score ┃ Matched Target              ┃ Encounter                   ┃ FHIR_Patient               ┃ ImagingStudy                                           ┃
┡━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1 │ 0.8500 │ DiagnosticReport            │ id: 128950                  │ id: 128861                 │ id: 128952 (available)                                 │
│   │        │ id: f9bc8f7b…               │ status/class: finished /    │ name: Mr. Dewey930         │ started: 2001-12-01                                    │
│   │        │ tag: CONCLUSION             │ IMP                         │ Carlos172 Nader710 PhD     │ modality: Computed Radiography                         │
│   │        │                             │ start: 2001-12-01           │ demo: male / 1926-08-21    │ proc: Plain X-ray of chest (procedure)                 │
│   │        │                             │ type: Hospital admission    │                            │ studyUid: 1.2.840.99999999.73980102.1007253...         │
│   │        │                             │ (procedure)                 │                            │ ---                                                    │
│   │        │                             │                             │                            │ id: 128953 (available)                                 │
│   │        │                             │                             │                            │ started: 2001-12-01                                    │
│   │        │                             │                             │                            │ modality: Ultrasound                                   │
│   │        │                             │                             │                            │ proc: Echocardiography (procedure)                     │
│   │        │                             │                             │                            │ studyUid: 1.2.840.99999999.71148084.1007253...         │
├───┼────────┼─────────────────────────────┼─────────────────────────────┼────────────────────────────┼────────────────────────────────────────────────────────┤
│ 2 │ 0.8489 │ DiagnosticReport            │ id: 490836                  │ id: 490797                 │ id: 490838 (available)                                 │
│   │        │ id: 402262a3…               │ status/class: finished /    │ name: Mr. Waylon572        │ started: 2014-12-03                                    │
│   │        │ tag: CONCLUSION             │ AMB                         │ Cory323 Emmerich580        │ modality: Ultrasound                                   │
│   │        │                             │ start: 2014-12-03           │ demo: male / 1959-07-15    │ proc: Transthoracic echocardiography (procedure)       │
│   │        │                             │ type: Encounter for problem │                            │ studyUid: 1.2.840.99999999.40697263.1417659...         │
│   │        │                             │ (procedure)                 │                            │                                                        │
├───┼────────┼─────────────────────────────┼─────────────────────────────┼────────────────────────────┼────────────────────────────────────────────────────────┤
│ 3 │ 0.8470 │ DiagnosticReport            │ id: 183773                  │ id: 182539                 │ id: 183821 (available)                                 │
│   │        │ id: 54f5d36f…               │ status/class: finished /    │ name: Mrs. Florrie453      │ started: 2016-07-08             

Traversal Coverage (5 hits): Encounter=5, FHIR_Patient=5, ImagingStudy=5 hits (6 total studies)

## 6. Multi-Query Batch Playground (Optional)

Supply a list of query phrases in `QUERY_PHRASES` to compare top results across multiple phrases using the exact same Section 2 parameter variables.

In [6]:
# --- Batch Query Variables ---
QUERY_PHRASES: list[str] = [
    "acute onset shortness of breath",
    "clear lungs with normal cardiac silhouette",
    "right lower lobe pneumonia",
]
BATCH_TOP_N: int = 3  # Number of top hits per query phrase to display in the summary table

batch_console = Console(width=CONSOLE_WIDTH)
batch_results: dict[str, list[SearchResult]] = {}
batch_table = Table(title=f"Batch Query Comparison (Top {BATCH_TOP_N} Hits per Phrase)", show_lines=True, expand=True)
batch_table.add_column("Query Phrase", style="cyan", ratio=1)
batch_table.add_column("#", justify="right", style="dim", no_wrap=True)
batch_table.add_column("Score", justify="right", style="green", no_wrap=True)
batch_table.add_column("Label", style="magenta", no_wrap=True)
batch_table.add_column("Field Tag", style="yellow", no_wrap=True)
batch_table.add_column("Text", style="white", ratio=3)
batch_table.add_column("Latency", justify="right", style="bright_blue", no_wrap=True)

for phrase in QUERY_PHRASES:
    t_start = time.perf_counter()
    hits = await search_svc.search_async(
        query=phrase,
        group_id=group_id,
        limit=LIMIT,
        min_score=MIN_SCORE,
        tag=TAG,
        include=INCLUDE,
        target_labels=TARGET_LABELS,
        target_field_tags=TARGET_FIELD_TAGS,
        include_deleted=INCLUDE_DELETED,
        test_id=TEST_ID,
        max_group_depth=MAX_GROUP_DEPTH,
    )
    ms = (time.perf_counter() - t_start) * 1000.0
    batch_results[phrase] = hits

    if not hits:
        batch_table.add_row(phrase, "-", "-", "-", "-", "[yellow](no hits)[/yellow]", f"{ms:.1f} ms")
        continue

    for rank, hit in enumerate(hits[:BATCH_TOP_N], start=1):
        batch_table.add_row(
            phrase if rank == 1 else "",
            str(rank),
            f"{hit.score:.4f}",
            hit.node_label or "-",
            hit.target_field_tag or "-",
            _truncate(hit.text, TEXT_PREVIEW_CHARS),
            f"{ms:.1f} ms" if rank == 1 else "",
        )

batch_console.print(batch_table)

2026-11-23 10:31:54 [info     ] Embedding text: task: search result | query: acute onset shortness of breath
2026-11-23 10:31:54 [info     ] Embedding text: task: search result | query: clear lungs with normal cardiac silhouette
2026-11-23 10:31:54 [info     ] Embedding text: task: search result | query: right lower lobe pneumonia


                                                         Batch Query Comparison (Top 3 Hits per Phrase)                                                         
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Query Phrase             ┃ # ┃  Score ┃ Label            ┃ Field Tag  ┃ Text                                                                      ┃  Latency ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ acute onset shortness of │ 1 │ 0.8500 │ DiagnosticReport │ CONCLUSION │ 2001-12-01 # Chief Complaint - Fatigue - Thirst - Chest Pressure - Nasal  │ 232.1 ms │
│ breath                   │   │        │                  │            │ Congestion # History of Present Illness Dewey930 is a 75 year-old         │          │
│                          │   │        │                  │            │ nonhispanic white male. # Social History Patient is single. Patient quit  │          │
│                          │   │        │                  │            │ smoki...                                                                  │          │
├──────────────────────────┼───┼────────┼──────────────────┼────────────┼───────────────────────────────────────────────────────────────────────────┼──────────┤
│                          │ 2 │ 0.8489 │ DiagnosticReport │ CONCLUSION │ 2014-12-03 # Chief Complaint - Shortness of Breath - Cough - Reduced Sex  │          │
│                          │   │        │                  │            │ Drive - urinary frequency # History of Present Illness Waylon572 is a 55  │          │
│                          │   │        │                  │            │ year-old nonhispanic white male. # Social History Patient is single.      │          │
│                          │   │        │                  │            │ Pa...                                                                     │          │
├──────────────────────────┼───┼────────┼──────────────────┼────────────┼───────────────────────────────────────────────────────────────────────────┼──────────┤
│                          │ 3 │ 0.8470 │ DiagnosticReport │ CONCLUSION │ 2016-07-08 # Chief Complaint No complaints. # History of Present Illness  │          │
│                          │   │        │                  │            │ Florrie453 Aja848 is a 81 year-old nonhispanic white female. Patient has  │          │
│                          │   │        │                  │            │ a history of gingival disease (disorder), medication review due           │          │
│                          │   │        │                  │            │ (situat...                                                                │          │
├──────────────────────────┼───┼────────┼──────────────────┼────────────┼───────────────────────────────────────────────────────────────────────────┼──────────┤
│ clear lungs with normal  │ 1 │ 0.8503 │ DiagnosticReport │ CONCLUSION │ 1993-12-17 # Chief Complaint No complaints. # History of Present Illness  │ 227.0 ms │
│ cardiac silhouette       │   │        │                  │            │ Rosario163 Saundra460 is a 27 year-old nonhispanic native female. Patient │          │
│                          │   │        │                  │            │ has a history of medication review due (situation), acute viral pharyn... │          │
├──────────────────────────┼───┼────────┼──────────────────┼────────────┼───────────────────────────────────────────────────────────────────────────┼──────────┤
│                          │ 2 │ 0.8498 │ DiagnosticReport │ CONCLUSION │ 1952-02-19 # Chief Complaint No complaints. # History of Present Illness  │          │
│                          │   │        │                  │            │ Heriberto162 is a 33 year-old nonhispanic asian male. Patient 

## 7. Inspect Full `SearchResult` Payload

Change `RESULT_INDEX` (0-based) or toggle `INSPECT_HYDRATED` to inspect either a hydrated hit from Section 5 (`hydrated_results`) or a base hit from Section 4 (`results`).

In [7]:
RESULT_INDEX: int = 0
INSPECT_HYDRATED: bool = True            # True = Section 5 hydrated_results; False = Section 4 results
EXCLUDE_RAW_FHIR_RESOURCE: bool = True   # Hide large inline raw FHIR JSON blobs inside hydrated nodes

source_name = "hydrated_results" if INSPECT_HYDRATED else "results"
inspect_list = globals().get(source_name) or []

exclude_spec = (
    {
        "encounter": {"resource"},
        "fhir_patient": {"resource"},
        "imaging_studies": {"__all__": {"resource"}},
    }
    if EXCLUDE_RAW_FHIR_RESOURCE
    else None
)

if not inspect_list:
    print(f"No results available in '{source_name}' to inspect.")
elif 0 <= RESULT_INDEX < len(inspect_list):
    print(inspect_list[RESULT_INDEX].model_dump_json(indent=2, exclude=exclude_spec))
else:
    print(f"RESULT_INDEX={RESULT_INDEX} is out of range for '{source_name}' (valid range: 0..{len(inspect_list) - 1}).")

{
  "score": 0.8500105142593384,
  "node_label": "DiagnosticReport",
  "node_id": "f9bc8f7b-8006-5278-9ff0-682fb05dbd30",
  "text": "2001-12-01\n\n# Chief Complaint\n- Fatigue\n- Thirst\n- Chest Pressure\n- Nasal Congestion\n\n\n# History of Present Illness\nDewey930 is a 75 year-old nonhispanic white male.\n\n# Social History\nPatient is single. Patient quit smoking at age 19.\n Patient identifies as heterosexual.\n\nPatient comes from a high socioeconomic background.\n Patient is a college graduate.\nPatient currently has Medicare.\n\n# Allergies\nNo Known Allergies.\n\n# Medications\nNo Active Medications.\n\n# Assessment and Plan\nPatient is presenting with history of coronary artery bypass grafting (situation). \n\n\n## Plan",
  "target_field_tag": "CONCLUSION",
  "model_tag": "embedding-v1",
  "embedding_id": "135abe0b-e58d-5094-a5c7-053107f385d7",
  "is_deleted": false,
  "encounter": {
    "id": "39e3449b-e53e-5bb0-be03-470919a56dd4",
    "test_id": null,
    "Action": "CREATE"

## 8. Cleanup

Run this cell when you are done experimenting to cleanly close the Neo4j driver connections.

In [ ]:
await neo4j_client.close()
print("Neo4j client connections closed.")